In [1]:
# ============================================================
# CELL 1 — Imports & Setup
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# NLP
import spacy
import nltk
from nltk.tokenize import word_tokenize

# Sklearn (for later clustering)
from sklearn.preprocessing import LabelEncoder

# Display settingsµ
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)

# Download NLTK data (only runs once)
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)

# Load spaCy model
nlp = spacy.load("en_core_web_sm")

print("All imports loaded successfully")

All imports loaded successfully


In [2]:
# ============================================================
# CELL 2 — Load Core Datasets (Reddit — 70%)
# ============================================================

import os

DATA_PATH = "../data/processed/"

# --- Load Reddit datasets ---
datasets = {
    "anxiety"      : "reddit_anxiety_cleaned.csv",
    "depression"   : "reddit_depression_cleaned.csv",
    "loneliness"   : "reddit_loneliness_cleaned.csv",
    "mentalhealth" : "reddit_mentalhealth_cleaned.csv",
    "therapy"      : "reddit_therapy_cleaned.csv",
}

dfs = []

for source, filename in datasets.items():
    path = os.path.join(DATA_PATH, filename)
    df = pd.read_csv(path)
    df["source"] = source          # tag where each row came from
    dfs.append(df)
    print(f"✅ {source:15s} → {len(df):>6,} rows | columns: {list(df.columns)}")

# --- Combine into one dataframe ---
reddit_df = pd.concat(dfs, ignore_index=True)

print(f"\n Combined Reddit dataset: {len(reddit_df):,} rows × {reddit_df.shape[1]} columns")
print(f"   Sources: {reddit_df['source'].value_counts().to_dict()}")


✅ anxiety         →    298 rows | columns: ['id', 'source', 'dataset', 'collected_at', 'language', 'cleaned_text', 'rating', 'label', 'emotion']
✅ depression      →    384 rows | columns: ['id', 'source', 'dataset', 'collected_at', 'language', 'cleaned_text', 'rating', 'label', 'emotion']
✅ loneliness      →    139 rows | columns: ['id', 'source', 'dataset', 'collected_at', 'language', 'cleaned_text', 'rating', 'label', 'emotion']
✅ mentalhealth    →    362 rows | columns: ['id', 'source', 'dataset', 'collected_at', 'language', 'cleaned_text', 'rating', 'label', 'emotion']
✅ therapy         →    170 rows | columns: ['id', 'source', 'dataset', 'collected_at', 'language', 'cleaned_text', 'rating', 'label', 'emotion']

 Combined Reddit dataset: 1,353 rows × 9 columns
   Sources: {'depression': 384, 'mentalhealth': 362, 'anxiety': 298, 'therapy': 170, 'loneliness': 139}


In [3]:
df.head()

,id,source,dataset,collected_at,language,cleaned_text,rating,label,emotion
0,1f788068-ff22-4584-99ee-29c72e9e483b,therapy,reddit_therapy,2026-05-02T23:23:51.791529+00:00,en,"Announcement - Important Update on Bot Advertisements Hello, all! We hope you're having all a lo...",NaN,NaN,NaN
1,8e8f54b5-fefc-4c84-99d3-c5777d0138fd,therapy,reddit_therapy,2026-05-02T23:23:51.793528+00:00,en,"Updated Rules Hello, r/therapy! We hope this post finds you well. We are writing to announce a m...",NaN,NaN,NaN
2,f89c07dc-1f71-4e6b-bb61-1d608edabd29,therapy,reddit_therapy,2026-05-02T23:23:51.793528+00:00,en,"""The ax forgets, but the forest remembers"" Recently I re-read the sentiment, ""The axe forgets bu...",NaN,NaN,NaN
3,822a6d67-2ed7-43ee-9cd6-6b81ebe09cf7,therapy,reddit_therapy,2026-05-02T23:23:51.793528+00:00,en,Not sure what type of therapist I'm looking for I've finally decided that I need to go through t...,NaN,NaN,NaN
4,64080621-f500-4f30-9909-d7cef2a8f4bf,therapy,reddit_therapy,2026-05-02T23:23:51.793528+00:00,en,"What should I do? I have a brother who is deaf-mute since birth, he is 26 years old now.. My mom...",NaN,NaN,NaN


In [4]:
# ============================================================
# CELL 3 — Load Support Dataset (Counseling — 20%)
# ============================================================

counseling_df = pd.read_csv(os.path.join(DATA_PATH, "hf_mh_counseling_amod_cleaned.csv"))
counseling_df["source"] = "counseling"

print(f"✅ counseling → {len(counseling_df):,} rows | columns: {list(counseling_df.columns)}")
counseling_df.head(2)

✅ counseling → 830 rows | columns: ['id', 'source', 'dataset', 'collected_at', 'language', 'cleaned_text', 'rating', 'label', 'emotion']


,id,source,dataset,collected_at,language,cleaned_text,rating,label,emotion
0,6aa92e1d-7ccc-4afa-bf2c-0e176b16f712,counseling,mh_counseling_amod,2026-05-03T10:40:00.425566+00:00,en,I'm going through some things with my feelings and myself. I barely sleep and I do nothing but t...,NaN,NaN,NaN
1,d0387c40-1d0d-4aa6-87c7-fb768b238e3b,counseling,mh_counseling_amod,2026-05-03T10:40:00.427774+00:00,en,"I have so many issues to address. I have a history of sexual abuse, I'm a breast cancer survivor...",NaN,NaN,NaN


In [5]:
# ============================================================
# CELL 4 — Explore & Preview
# ============================================================

# --- Reddit overview ---
print("=" * 55)
print("📊 REDDIT DATASETS")
print("=" * 55)
print(reddit_df["source"].value_counts())
print(f"\nTotal rows      : {len(reddit_df):,}")
print(f"Null text rows  : {reddit_df['cleaned_text'].isna().sum():,}")
print(f"Avg text length : {reddit_df['cleaned_text'].dropna().str.len().mean():.0f} chars")

# --- Drop null text rows ---
reddit_df = reddit_df.dropna(subset=["cleaned_text"]).reset_index(drop=True)
print(f"\n✅ After dropping nulls: {len(reddit_df):,} rows remain")

# --- Quick sample ---
print("\n--- Sample post (anxiety) ---")
sample = reddit_df[reddit_df["source"] == "anxiety"]["cleaned_text"].iloc[0]
print(sample[:300], "...")

📊 REDDIT DATASETS
source
depression      384
mentalhealth    362
anxiety         298
therapy         170
loneliness      139
Name: count, dtype: int64

Total rows      : 1,353
Null text rows  : 0
Avg text length : 925 chars

✅ After dropping nulls: 1,353 rows remain

--- Sample post (anxiety) ---
Recruiting Moderators! Hey friends, We are looking to grow the team again here on our lovely subreddit. If you are interested, please fill out the form on our application page for r/Anxiety. If you have any questions, feel free to drop them on this post or send us a modmail. Thanks! ...


In [6]:
# ============================================================
# CELL 5 — Clean & Filter Noise
# ============================================================

# --- Noise patterns to drop (mod posts, announcements, etc.) ---
noise_keywords = [
    "recruiting moderators", "mod announcement", "subreddit rules",
    "updated rules", "bot advertisement", "automoderator",
    "weekly thread", "daily thread", "megathread"
]

def is_noise(text):
    text_lower = text.lower()
    return any(kw in text_lower for kw in noise_keywords)

# --- Apply filter ---
before = len(reddit_df)
reddit_df = reddit_df[~reddit_df["cleaned_text"].apply(is_noise)].reset_index(drop=True)
after = len(reddit_df)

print(f"  Removed {before - after} noise rows")
print(f" Clean Reddit rows: {after:,}")

# --- Also filter very short posts (less than 50 chars — likely useless) ---
reddit_df = reddit_df[reddit_df["cleaned_text"].str.len() >= 50].reset_index(drop=True)
print(f" After length filter: {len(reddit_df):,} rows remain")

# --- Combine Reddit + Counseling into master df ---
master_df = pd.concat([reddit_df, counseling_df], ignore_index=True)
master_df = master_df[["id", "source", "cleaned_text"]].copy()

print(f"\nMaster dataframe: {len(master_df):,} rows")
print(f"   Sources: {master_df['source'].value_counts().to_dict()}")

  Removed 4 noise rows
 Clean Reddit rows: 1,349
 After length filter: 1,307 rows remain

Master dataframe: 2,137 rows
   Sources: {'counseling': 830, 'depression': 364, 'mentalhealth': 359, 'anxiety': 290, 'therapy': 166, 'loneliness': 128}


In [7]:
from sentence_transformers import SentenceTransformer, util
import torch

model = SentenceTransformer("all-MiniLM-L6-v2")

categories = [
    "anxiety and panic attacks",
    "depression and hopelessness",
    "loneliness and social isolation",
    "trauma and abuse",
    "relationship and family issues",
    "low self worth and self esteem",
    "suicidal thoughts and self harm",
    "sleep problems and exhaustion",
    "work stress and burnout",
    "academic pressure and school stress",
    "anger and emotional dysregulation",
    "intrusive thoughts and paranoia",
    "therapy seeking and looking for help",
    "general mental health",
]

category_embeddings = model.encode(categories, convert_to_tensor=True)
print(f"✅ Model loaded | {len(categories)} categories embedded")

c:\Users\Bichr\miniconda3\envs\data_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 785.94it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Model loaded | 14 categories embedded


In [8]:
def classify_pain_points(text, threshold=0.30, top_k=3):
    text_embedding = model.encode(str(text)[:512], convert_to_tensor=True)
    scores = util.cos_sim(text_embedding, category_embeddings)[0]
    
    # pick all labels above threshold, up to top_k
    hits = [(categories[i], float(scores[i])) for i in range(len(categories)) if scores[i] >= threshold]
    hits = sorted(hits, key=lambda x: -x[1])[:top_k]
    
    if not hits:
        return ["general mental health"]
    return [label for label, score in hits]

master_df["pain_points"] = master_df["cleaned_text"].apply(classify_pain_points)
master_df["pain_point_count"] = master_df["pain_points"].apply(len)

print("📊 Pain point distribution:")
from collections import Counter
all_pain_points = [pp for sublist in master_df["pain_points"] for pp in sublist]
pain_counts = Counter(all_pain_points)
for pain, count in pain_counts.most_common():
    pct = count / len(master_df) * 100
    print(f"   {pain:40s} → {count:>4} rows ({pct:.1f}%)")

print(f"\n✅ Avg pain points/row : {master_df['pain_point_count'].mean():.2f}")

📊 Pain point distribution:
   general mental health                    →  777 rows (36.4%)
   therapy seeking and looking for help     →  671 rows (31.4%)
   depression and hopelessness              →  544 rows (25.5%)
   anxiety and panic attacks                →  368 rows (17.2%)
   intrusive thoughts and paranoia          →  288 rows (13.5%)
   loneliness and social isolation          →  213 rows (10.0%)
   suicidal thoughts and self harm          →  175 rows (8.2%)
   anger and emotional dysregulation        →  157 rows (7.3%)
   relationship and family issues           →  126 rows (5.9%)
   trauma and abuse                         →  117 rows (5.5%)
   sleep problems and exhaustion            →  116 rows (5.4%)
   low self worth and self esteem           →  100 rows (4.7%)
   work stress and burnout                  →   81 rows (3.8%)
   academic pressure and school stress      →   37 rows (1.7%)

✅ Avg pain points/row : 1.76


In [9]:
macro_map = {
    "emotional distress"     : ["depression and hopelessness", "low self worth and self esteem", "anger and emotional dysregulation"],
    "anxiety & fear"         : ["anxiety and panic attacks", "intrusive thoughts and paranoia"],
    "social & relationships" : ["loneliness and social isolation", "relationship and family issues"],
    "trauma & crisis"        : ["trauma and abuse", "suicidal thoughts and self harm"],
    "life stressors"         : ["work stress and burnout", "academic pressure and school stress", "sleep problems and exhaustion"],
    "help seeking"           : ["therapy seeking and looking for help", "general mental health"],
}

sub_to_macro = {sub: macro for macro, subs in macro_map.items() for sub in subs}
priority = ["trauma & crisis", "emotional distress", "anxiety & fear", "social & relationships", "life stressors", "help seeking"]

def assign_macro(pain_points):
    macros = set(sub_to_macro.get(pp, "help seeking") for pp in pain_points)
    return next((m for m in priority if m in macros), "help seeking")

master_df["primary_macro"] = master_df["pain_points"].apply(assign_macro)

print("📊 Macro group distribution:")
for macro, count in master_df["primary_macro"].value_counts().items():
    pct = count / len(master_df) * 100
    print(f"   {macro:25s} → {count:>4} ({pct:.1f}%) {'█' * int(pct/2)}")

📊 Macro group distribution:
   help seeking              →  779 (36.5%) ██████████████████
   emotional distress        →  551 (25.8%) ████████████
   anxiety & fear            →  318 (14.9%) ███████
   trauma & crisis           →  279 (13.1%) ██████
   social & relationships    →  160 (7.5%) ███
   life stressors            →   50 (2.3%) █
